# Pitch Usage Experiments

Three non-traditional visualizations to replace the standard horizontal bar chart: **Treemap**, **Radial Gauge**, and **Sequencing Barcode**.

## Step 1: Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import squarify
from pathlib import Path

# Mallitalytics brand palette (from pitcher_card_gemini.py)
PALETTE = {
    "card_bg":       "#131B23", 
    "header_bg":     "#1C2836",
    "panel_bg":      "#1A2430",
    "table_bg":      "#16202A",
    "table_alt":     "#1C2836",
    "text_primary":  "#F5F2ED",
    "text_secondary":"#A8BDD0",
    "text_lo":       "#5D7A93",
    "accent_orange": "#E8712B",
    "accent_green":  "#66BB6A",
    "accent_red":    "#E74C3C",
    "grid":          "#2C3E50",
    "border":        "#2E4A62",
    "zone_edge":     "#8FA3B8",
}

PITCH_COLOURS = {
    'FF': {'colour': '#FF007D', 'name': '4-Seam'},
    'SI': {'colour': '#98165D', 'name': 'Sinker'},
    'FC': {'colour': '#BE5FA0', 'name': 'Cutter'},
    'CH': {'colour': '#F79E70', 'name': 'Changeup'},
    'FS': {'colour': '#FE6100', 'name': 'Splitter'},
    'SL': {'colour': '#67E18D', 'name': 'Slider'},
    'ST': {'colour': '#1BB999', 'name': 'Sweeper'},
    'CU': {'colour': '#3025CE', 'name': 'Curveball'},
    'KC': {'colour': '#311D8B', 'name': 'Knuck. Curve'},
    'UN': {'colour': '#9C8975', 'name': 'Unknown'},
}
DICT_COLOUR = {k: v['colour'] for k, v in PITCH_COLOURS.items()}
DICT_PITCH  = {k: v['name']   for k, v in PITCH_COLOURS.items()}

def load_game(parquet_path, pitcher_id):
    df = pd.read_parquet(parquet_path)
    col = "pitcher" if "pitcher" in df.columns else "pitcher_id"
    df = df[df[col] == pitcher_id].copy()
    if df.empty:
        raise ValueError(f"Pitcher {pitcher_id} not found.")
    return df

def process_pitches(df):
    swing_codes = ['foul_bunt','foul','hit_into_play','swinging_strike','foul_tip','swinging_strike_blocked','missed_bunt','bunt_foul_tip']
    whiff_codes = ['swinging_strike','foul_tip','swinging_strike_blocked']
    df = df.copy()
    df['swing']     = df['description'].isin(swing_codes)
    df['whiff']     = df['description'].isin(whiff_codes)
    df['in_zone']   = df['zone'] < 10
    df['out_zone']  = df['zone'] > 10
    df['chase']     = (~df['in_zone']) & df['swing']
    df['is_strike'] = df['type'] == 'S'
    df['pfx_z_in']  = df['pfx_z'] * 12
    df['pfx_x_in']  = df['pfx_x'] * 12
    if 'launch_speed' in df.columns:
        df['hard_hit'] = df['launch_speed'] >= 95.0
    else:
        df['hard_hit'] = False
    if 'estimated_woba_using_speedangle' in df.columns:
        df['is_damage'] = (df['hard_hit']) | (df['estimated_woba_using_speedangle'] >= 0.350)
    else:
        df['is_damage'] = df['hard_hit']
    return df

In [ ]:
PARQUET_PATH = Path("/Users/gilrojasb/Desktop/Mallitalytics_VS/MLB/data/warehouse/mlb/2025/regular_season/pitches_enriched/game_776136_20250928_pitches_enriched.parquet")
PITCHER_ID = 676664

df_raw = load_game(str(PARQUET_PATH), PITCHER_ID)
df = process_pitches(df_raw)

# Ensure we have pitch_type and stand (batter stance)
assert 'pitch_type' in df.columns and 'stand' in df.columns, "Need pitch_type and stand"
df['pitch_type'] = df['pitch_type'].fillna('UN').astype(str)
df['stand'] = df['stand'].fillna('R').astype(str)

print(f"Loaded {len(df)} pitches. Pitch types: {df['pitch_type'].unique().tolist()}")
print(f"vs LHB: {(df['stand']=='L').sum()}, vs RHB: {(df['stand']=='R').sum()}")

## Step 2: Option 1 — The Arsenal Treemap

Side-by-side treemap: **left = vs LHB**, **right = vs RHB**. Rectangle size = pitch frequency; colors from `DICT_COLOUR`.

In [ ]:
fig, (ax_l, ax_r) = plt.subplots(1, 2, figsize=(12, 6))
fig.patch.set_facecolor(PALETTE["card_bg"])

def plot_treemap_side(ax, sub_df, title):
    ax.set_facecolor(PALETTE["panel_bg"])
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    for spine in ax.spines.values():
        spine.set_visible(False)
    if sub_df.empty:
        ax.text(0.5, 0.5, "No pitches", color=PALETTE["text_secondary"], ha='center', va='center', transform=ax.transAxes)
        return
    counts = sub_df.groupby('pitch_type').size().reindex(sub_df['pitch_type'].unique()).fillna(0).astype(int)
    counts = counts[counts > 0]
    if counts.empty:
        return
    labels = [DICT_PITCH.get(pt, pt) for pt in counts.index]
    colors = [DICT_COLOUR.get(pt, '#9C8975') for pt in counts.index]
    sizes = counts.values
    norm = squarify.normalize_sizes(sizes, 1, 1)
    rects = squarify.squarify(norm, 0, 0, 1, 1)
    for r, pt, color in zip(rects, counts.index, colors):
        rect = mpatches.Rectangle((r['x'], r['y']), r['dx'], r['dy'], facecolor=color, edgecolor=PALETTE['border'], linewidth=0.8)
        ax.add_patch(rect)
        pct = counts[pt] / counts.sum() * 100
        ax.text(r['x'] + r['dx']/2, r['y'] + r['dy']/2, f"{DICT_PITCH.get(pt, pt)}\n{pct:.0f}%",
                color='#111111' if (sum(mpl.colors.to_rgb(color)) / 3) > 0.5 else '#FFFFFF',
                ha='center', va='center', fontsize=9, fontweight='bold')
    ax.set_title(title, color=PALETTE["text_primary"], fontsize=12, fontweight='bold', pad=8)

df_l = df[df['stand'] == 'L']
df_r = df[df['stand'] == 'R']
plot_treemap_side(ax_l, df_l, "vs LHB")
plot_treemap_side(ax_r, df_r, "vs RHB")
plt.suptitle("Pitch Usage — Arsenal Treemap", color=PALETTE["text_primary"], fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## Step 3: Option 2 — The Radial Usage Gauge

Semi-circle (180°) polar plot: **inner ring = vs LHB**, **outer ring = vs RHB**. Radial bar length = usage %. High-tech dashboard style.

In [ ]:
from math import pi

pitch_types = df['pitch_type'].value_counts().index.tolist()
n = len(pitch_types)
if n == 0:
    n = 1
    pitch_types = ['UN']

df_l = df[df['stand'] == 'L']
df_r = df[df['stand'] == 'R']
total_l = len(df_l) or 1
total_r = len(df_r) or 1
pct_l = [len(df_l[df_l['pitch_type'] == pt]) / total_l for pt in pitch_types]
pct_r = [len(df_r[df_r['pitch_type'] == pt]) / total_r for pt in pitch_types]

fig = plt.figure(figsize=(10, 6))
fig.patch.set_facecolor(PALETTE["card_bg"])
ax = fig.add_subplot(111, projection='polar')
ax.set_facecolor(PALETTE["panel_bg"])

# Semi-circle: use only 0 to pi (180°)
theta = np.linspace(0, np.pi, n + 1)
width = (np.pi / n) * 0.85

# Inner ring (LHB) — radius 0.25 to 0.25 + pct
for i, pt in enumerate(pitch_types):
    r_inner, r_outer = 0.25, 0.25 + 0.35 * pct_l[i]
    t = [theta[i], theta[i], theta[i+1], theta[i+1]]
    r = [r_inner, r_outer, r_outer, r_inner]
    ax.fill(t, r, facecolor=DICT_COLOUR.get(pt, '#9C8975'), edgecolor=PALETTE['border'], linewidth=0.6, alpha=0.9)

# Outer ring (RHB) — radius 0.65 to 0.65 + pct
for i, pt in enumerate(pitch_types):
    r_inner, r_outer = 0.65, 0.65 + 0.30 * pct_r[i]
    t = [theta[i], theta[i], theta[i+1], theta[i+1]]
    r = [r_inner, r_outer, r_outer, r_inner]
    ax.fill(t, r, facecolor=DICT_COLOUR.get(pt, '#9C8975'), edgecolor=PALETTE['border'], linewidth=0.6, alpha=0.9)

ax.set_theta_offset(pi / 2)
ax.set_theta_direction(-1)
ax.set_thetamin(0)
ax.set_thetamax(180)
ax.set_ylim(0, 1.0)
ax.set_yticks([0.25, 0.65])
ax.set_yticklabels(['vs LHB', 'vs RHB'], color=PALETTE['text_secondary'], fontsize=9)
ax.set_xticks(theta[:-1] + (theta[1] - theta[0]) / 2)
ax.set_xticklabels([DICT_PITCH.get(pt, pt) for pt in pitch_types], color=PALETTE['text_primary'], fontsize=9)
ax.spines['polar'].set_color(PALETTE['border'])
ax.grid(True, color=PALETTE['grid'], alpha=0.4, linestyle='-')
ax.set_title("Pitch Usage — Radial Gauge", color=PALETTE["text_primary"], fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

## Step 4: Option 3 — The Sequencing Barcode

**X** = pitch number (chronological). **Y** = two tracks: **top = vs LHB**, **bottom = vs RHB**. Each vertical line = one pitch, colored by `pitch_type`. DNA-barcode style, minimalist axes.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
fig.patch.set_facecolor(PALETTE["card_bg"])
ax.set_facecolor(PALETTE["panel_bg"])

for spine in ax.spines.values():
    spine.set_visible(False)
ax.set_xticks([])
ax.set_yticks([])

# Two tracks: 1 = vs LHB (top), 0 = vs RHB (bottom)
df = df.reset_index(drop=True)
df['pitch_num'] = np.arange(len(df))
df['track'] = (df['stand'] == 'L').astype(int)

y_center = {0: 0.25, 1: 0.75}
line_h = 0.08

for _, row in df.iterrows():
    x = row['pitch_num']
    y = y_center[row['track']]
    color = DICT_COLOUR.get(row['pitch_type'], '#9C8975')
    ax.vlines(x, y - line_h/2, y + line_h/2, colors=color, linewidth=1.2, alpha=0.95)

ax.set_xlim(-0.5, len(df) - 0.5)
ax.set_ylim(0, 1)
ax.text(-0.02 * len(df), 0.75, "vs LHB", color=PALETTE['text_secondary'], fontsize=10, ha='right', va='center')
ax.text(-0.02 * len(df), 0.25, "vs RHB", color=PALETTE['text_secondary'], fontsize=10, ha='right', va='center')
ax.set_title("Pitch Usage — Sequencing Barcode", color=PALETTE["text_primary"], fontsize=14, fontweight='bold', pad=10)
plt.tight_layout()
plt.show()